# Generative AI + GraphRAG Demo
## House/Apartment Adjacency (Kùzu-based, No-Geometry, Global-Candidate Builder)

In [ ]:
# You don't need to run this cell if you have pip installed topologicpy
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

In [ ]:
# --- TopologicPy imports ---
from topologicpy.Vertex import Vertex
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Kuzu import Kuzu
from topologicpy.Graph import Graph

In [ ]:
"""
GraphRAG Demo — House/Apartment Adjacency (Kùzu-based, No-Geometry, Global-Candidate Builder)
=====================================================================================

**What this is**
A compact, Jupyter-friendly demo that:
1) Reads *TopologicPy-like* graphs (JSON) from a folder.
2) Builds *topological* graphs strictly from the file's **vertices** and **edges** (ignores geometry entirely).
3) Loads them into a Kùzu DB using your `Kuzu.py` schema (Graph, Vertex, Edge).
4) **New logic:** At each iteration, we:
   - Build a **global candidate list** of neighbor labels by querying *all graphs* for the labels present in the **currently built graph** (frequency-ranked).
   - Ask the LLM to pick **one** action: either
     - **ADD** a node (may choose from the list or propose a new label not in the list) and connect it to a chosen existing node, or
     - **CONNECT** two existing nodes (no new node).
   - Apply the action to the **working graph** (we create/update it in the DB).
   - Save a full-graph snapshot and repeat until a stopping rule is met.

Notes
-----
- We *ignore* any polygon/geometry in JSON and rely solely on `vertices` and `edges`.
- Vertices get `label` from `node_name` or `roomtype` if present; fallback to vertex id.
- `x,y,z` default to 0.0 if missing. Original vertex/edge dicts preserved in `props` JSON.
- If `OPENAI_API_KEY` is not set or OpenAI SDK is unavailable, a deterministic heuristic is used so the demo still runs.
- Edge suggestions, when accepted, are inserted with label `"suggested"` (bidirectional for simplicity).
- **Requested enhancement:** the seed node now **copies props (and x,y,z if present)** from the best-matching example across all graphs.

"""
from __future__ import annotations
import os, json, glob
from dataclasses import dataclass
from typing import List, Dict, Any, Optional, Tuple
from collections import Counter

# --- Kùzu manager (ensure Kuzu.py is on sys.path or in the same directory) ---
from topologicpy.Kuzu import Kuzu

# --- Optional OpenAI (used only if available + key set) ---
try:
    import openai  # type: ignore
except Exception:
    openai = None

# --- Optional TopologicPy for snapshots -> real Graph objects ---
try:
    from topologicpy.Graph import Graph as TPGraph
    from topologicpy.Vertex import Vertex as TPVertex
    from topologicpy.Edge import Edge as TPEdge
    from topologicpy.Dictionary import Dictionary as TPDict
    from topologicpy.Topology import Topology as TPTopology
    _TOPOLOGICPY_AVAILABLE = True
except Exception:
    _TOPOLOGICPY_AVAILABLE = False

# ---------------------
# Data models
# ---------------------
@dataclass
class Vtx:
    id: str
    label: str
    x: float
    y: float
    z: float
    props: Dict[str, Any]

@dataclass
class ERel:
    src: str
    dst: str
    label: str
    props: Dict[str, Any]

# ---------------------
# JSON → (Vertices, Edges)
# ---------------------

def load_topologic_graph(path: str, labelKey: str) -> tuple[list[Vtx], list[ERel]]:
    """Load a TopologicPy-like graph JSON. We ignore geometry; we only use vertices and edges dictionaries.
    Expected (flexible) shape:
    {
      "vertices": {
          "Vertex_0000": {"node_name": "Entrance", "x": 1.2, "y": 3.4, ...},
          ...
      },
      "edges": {
          "Edge_00": {"source": "Vertex_0000", "target": "Vertex_0004", "connectivity": "door", ...},
          ...
      }
    }
    """
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    raw_vs: Dict[str, Dict[str, Any]] = data.get("vertices", {}) or {}
    raw_es: Dict[str, Dict[str, Any]] = data.get("edges", {}) or {}

    vertices: list[Vtx] = []
    for vid, v in raw_vs.items():
        label = v.get(labelKey, str(vid))
        x = float(v.get("x", 0.0))
        y = float(v.get("y", 0.0))
        z = float(v.get("z", 0.0))
        vertices.append(Vtx(id=vid, label=str(label), x=x, y=y, z=z, props=v))

    edges: list[ERel] = []
    for eid, e in raw_es.items():
        src = str(e.get("source"))
        dst = str(e.get("target"))
        label = str(e.get("connectivity") or e.get("label") or "adjacent")
        if not src or not dst:
            continue
        edges.append(ERel(src=src, dst=dst, label=label, props=e))

    return vertices, edges

# ---------------------
# Kùzu helpers
# ---------------------

def ensure_schema(manager):
    Kuzu.EnsureSchema(manager, silent=False)


def upsert_graph(manager, graph_id: str, vertices: list[Vtx], edges: list[ERel], undirected: bool):
    """Insert a graph with vertices/edges into Kùzu using raw Cypher.
    If undirected=True, we create two directed edges for each input edge.
    """
    ensure_schema(manager)

    # Clear prior graph with same id
    manager.exec("MATCH (a:Vertex)-[r:Edge]->(b:Vertex) WHERE a.graph_id=$gid AND b.graph_id=$gid DELETE r;",
                 {"gid": graph_id}, write=True)
    manager.exec("MATCH (v:Vertex) WHERE v.graph_id=$gid DELETE v;", {"gid": graph_id}, write=True)
    manager.exec("MATCH (g:Graph) WHERE g.id=$id DELETE g;", {"id": graph_id}, write=True)

    # Create Graph card
    manager.exec(
        """
        CREATE (g:Graph {id:$id, label:$label, num_nodes:$n, num_edges:$m, props:$props});
        """,
        {"id": graph_id, "label": graph_id, "n": len(vertices), "m": len(edges), "props": json.dumps({})},
        write=True,
    )

    # Insert vertices

    query = """
    CREATE (v:Vertex {
        id:$id,
        graph_id:$gid,
        label:$label,
        x:$x,
        y:$y,
        z:$z,
        props:$props
    });
    """

    for v in vertices:
        params = {
                "id": f"{graph_id}:{v.id}",
                "gid": graph_id,
                "label": v.label,
                "x": float(v.x),
                "y": float(v.y),
                "z": float(v.z),
                "props": json.dumps(v.props),
                }
        manager.exec(query, params, write=True)
    # Insert edges (directed; if undirected, add reverse)
    for e in edges:
        params = {"a": f"{graph_id}:{e.src}", "b": f"{graph_id}:{e.dst}", "label": e.label, "props": json.dumps(e.props)}
        manager.exec(
            """
            MATCH (va:Vertex {id:$a}), (vb:Vertex {id:$b})
            CREATE (va)-[:Edge {label:$label, props:$props}]->(vb);
            """,
            params,
            write=True,
        )
        if undirected:
            manager.exec(
                """
                MATCH (va:Vertex {id:$a}), (vb:Vertex {id:$b})
                CREATE (vb)-[:Edge {label:$label, props:$props}]->(va);
                """,
                params,
                write=True,
            )

# --- Small builders for the *working* graph we are constructing ---

def create_graph_card_if_missing(manager, graph_id: str):
    rows = manager.exec("MATCH (g:Graph {id:$id}) RETURN 1 LIMIT 1", {"id": graph_id}, write=False) or []
    if not rows:
        manager.exec(
            "CREATE (g:Graph {id:$id, label:$id, num_nodes:0, num_edges:0, props:'{}'})",
            {"id": graph_id}, write=True)

def aabb(graph):
    vertices = Graph.Vertices(graph)
    x_coords = [Vertex.X(v) for v in vertices]
    y_coords = [Vertex.Y(v) for v in vertices]
    x_min = min(x_coords)
    x_max = max(x_coords)
    y_min = min(y_coords)
    y_max = max(y_coords)
    return x_min, y_min, x_max, y_max

def create_vertex(graph, id: str = None, label: str = "Unknown", vertexLabelKey: str = "label", props: dict = {}, x: float = None, y: float = None, z: float = None):
    import random
    x_min, y_min, x_max, y_max = aabb(graph)
    if x is None:
        x = random.uniform(x_min, x_max)
    if y is None:
        y = random.uniform(y_min, y_max)
    if z is None:
        z = 0
    v = Vertex.ByCoordinates(x, y, z)
    if id == None:
        id = unique_id(graph)
    d = Dictionary.ByPythonDictionary(props)
    d = Dictionary.SetValuesAtKeys(d, ["id", vertexLabelKey], [id, label])
    v = Topology.SetDictionary(v, d)
    graph = Graph.AddVertex(graph, v)
    return graph

def delete_vertex(graph, vertexID: str) -> bool:
    """
    Delete a vertex (and all incident edges) from a given graph
    """
    vertices = Graph.Vertices(graph)
    status = False
    for v in vertices:
        d = Topology.Dictionary(v)
        v_id = Dictionary.ValueAtKey(d, "id", "Unknown")
        if v_id == vertexID:
            Graph.RemoveVertex(graph, v)
            return True
    return False

def vertex_by_id(graph, id):
    vertices = Graph.Vertices(graph)
    for v in vertices:
        d = Topology.Dictionary(v)
        v_id = Dictionary.ValueAtKey(d, "id", "Unknown")
        if v_id == id:
            return v
    return None

def edge_exists(graph, a: str, b: str) -> bool:
    if a == b:
        return False
    edges = Graph.Edges(graph)
    for edge in edges:
        d = Topology.Dictionary(edge)
        src = Dictionary.ValueAtKey(d, "src", "Unknown")
        dst = Dictionary.ValueAtKey(d, "dst", "Unknown")
        if a in [src, dst] and b in [src, dst]:
            return True
    return False

def create_edge(graph, a: str, b: str, label: str = "suggested"):
    from topologicpy.Edge import Edge
    
    if edge_exists(graph, a, b):
        print("Warning: Edge already exists. Skipping.")
        return False
    v_a = vertex_by_id(graph, a)
    v_b = vertex_by_id(graph, b)
    if v_a and v_b:
        e = Edge.ByStartVertexEndVertex(v_a, v_b)
        d = Dictionary.ByKeysValues(["src", "dst", "label"], [a, b, label])
        e = Topology.SetDictionary(e, d)
        return_graph = Graph.AddEdge(graph, e)
        return return_graph
    return None
    

def delete_edge(graph, a: str, b: str):
    """
    Deletes an edge between two vertices if it exists.
    """
    if a == b:
        return False
    edges = Graph.Edges(graph)
    for e in edges:
        d = Topology.Dictionary(v)
        src_id = Dictionary.ValueAtKey(d, "src")
        dst_id = Dictionary.ValueAtKey(d, "dst")
        if src_id in [a,b] and dst_id in [a,b]:
            Graph.RemoveEdge(graph, e)
            return True
    return False

def unique_id(graph):
    vertices = Graph.Vertices(graph)
    ids = [int(Dictionary.ValueAtKey(Topology.Dictionary(v), "id")) for v in vertices]
    max_id = max(ids)
    return str(max_id+1)

def list_working_nodes_edges(graph, vertexLabelKey):
    edges = Graph.Edges(graph)
    vertices = Graph.Vertices(graph)
    return_vertices = []
    for v in vertices:
        d = Topology.Dictionary(v)
        id = Dictionary.ValueAtKey(d, "id")
        label = Dictionary.ValueAtKey(d, vertexLabelKey)
        return_vertices.append({"id": id, vertexLabelKey: label})
    return_edges = []
    for i, e in enumerate(edges):
        d = Topology.Dictionary(e)
        src = Dictionary.ValueAtKey(d, "src", "Unknown")
        dst = Dictionary.ValueAtKey(d, "dst", "Unknown")
        return_edges.append({"src": src, "dst": dst})
    return return_vertices, return_edges

def max_neighbors_for_label(
    manager,
    label_query: str,
    *,
    undirected: bool = True,
    substring: bool = True,
    selection_mode: str = "first",            # "first" | "max_out_degree_within_graph"
    include_graph_ids: set[str] | None = None,
    exclude_graph_ids_prefixes: tuple[str, ...] = ("work_",),
    exclude_edge_labels: set[str] = frozenset({"suggested"})
) -> int:
    """
    Return the maximum number of unique neighbors for the label across the DB,
    but constrain to **one representative vertex per graph** to avoid inflation
    when a graph contains multiple vertices with the same label.

    Parameters mirror the earlier function, with `selection_mode` controlling how
    the per-graph representative is chosen:
      - "first": smallest v.id (stable & fast)
      - "max_out_degree_within_graph": pick the matching vertex that has the
        highest out-degree (using filtered edges), then count its neighbors
        (undirected or directed per `undirected` flag).

    Returns
    -------
    int
    """

    # --- Fetch vertices
    rows_v = manager.exec(
        """
        MATCH (v:Vertex)
        RETURN v.id AS id, v.graph_id AS gid, v.label AS label
        """,
        {}, write=False
    ) or []

    def graph_allowed(gid: str) -> bool:
        if include_graph_ids is not None:
            return gid in include_graph_ids
        return not any(gid.startswith(pref) for pref in exclude_graph_ids_prefixes)

    needle = (label_query or "").strip().lower()
    # group matching vertices by graph id
    matches_by_gid: dict[str, list[str]] = {}
    labels_by_vid: dict[str, str] = {}

    for r in rows_v:
        vid = r["id"]
        gid = r.get("gid", "")
        lbl = str(r.get("label") or "")
        if not graph_allowed(gid):
            continue
        labels_by_vid[vid] = lbl
        lbln = lbl.strip().lower()
        ok = (needle in lbln) if substring else (lbln == needle)
        if ok:
            matches_by_gid.setdefault(gid, []).append(vid)

    if not matches_by_gid:
        return -1

    # --- Fetch edges once (filtered)
    rows_e = manager.exec(
        """
        MATCH (a:Vertex)-[r:Edge]->(b:Vertex)
        RETURN a.id AS a, a.graph_id AS agid, b.id AS b, b.graph_id AS bgid, r.label AS rlabel
        """,
        {}, write=False
    ) or []

    edges = []
    for r in rows_e:
        a, agid = r["a"], r.get("agid", "")
        b, bgid = r["b"], r.get("bgid", "")
        if not (graph_allowed(agid) and graph_allowed(bgid)):
            continue
        if str(r.get("rlabel") or "") in exclude_edge_labels:
            continue
        edges.append((a, b))

    # Pre-index neighbors for quick degree checks
    out_neighbors: dict[str, set[str]] = {}
    in_neighbors: dict[str, set[str]] = {}
    for a, b in edges:
        out_neighbors.setdefault(a, set()).add(b)
        in_neighbors.setdefault(b, set()).add(a)

    # Choose exactly one representative per graph
    rep_ids: set[str] = set()
    if selection_mode == "max_out_degree_within_graph":
        for gid, vids in matches_by_gid.items():
            # pick the vertex with max OUT-degree (based on filtered edges)
            best_vid = max(vids, key=lambda v: len(out_neighbors.get(v, set())))
            rep_ids.add(best_vid)
    else:  # "first" (deterministic by min id)
        for gid, vids in matches_by_gid.items():
            rep_ids.add(min(vids))

    # Compute neighbor counts for representatives only
    def neighbor_set(v: str) -> set[str]:
        if undirected:
            return out_neighbors.get(v, set()) | in_neighbors.get(v, set())
        return out_neighbors.get(v, set())

    return max((len(neighbor_set(v)) for v in rep_ids), default=0)



# ---------------------
# Global candidate list (across *all* graphs)
# ---------------------

def _anchor_labels_with_degree_cap(manager, graph, vertexLabelKey: str = "label") -> list[str]:
    vertices = Graph.Vertices(graph)

    anchor_labels = []
    print(f" The following nodes have less connections than the maximum found in the graph database so they will be considered for expansion:")
    for v in vertices:
        deg = Graph.VertexDegree(graph, v)
        d = Topology.Dictionary(v)
        anchor_label = Dictionary.ValueAtKey(d, vertexLabelKey, "")
        max_degree = max_neighbors_for_label(manager, anchor_label) 
        if max_degree < 0:
            print(f"  . {anchor_label} (No. Connections: {deg}, Not found in DB) ")
            anchor_labels.append(anchor_label)
        elif deg < max_degree:
            print(f"  . {anchor_label} (No. Connections: {deg}, Max found in DB: {max_degree}) ")
            anchor_labels.append(anchor_label)
        else:
            continue

    # Clean, dedupe, keep non-empty
    return sorted({lbl for lbl in anchor_labels if lbl})


def fetch_all_pairs(manager, graph, vertexLabelKey: str = "label") -> list[tuple[str, str]]:
    """
    Enhanced: Use ONLY input nodes (anchors) from the current working graph whose undirected degree ≤ n,
    then fetch (a.label, b.label) pairs from the *entire* dataset, filtered to those anchors.
    Returns list of (a_label, b_label) pairs.
    """
    # 1) Get anchor labels from the working graph, filtered by degree cap
    anchors = _anchor_labels_with_degree_cap(manager, graph, vertexLabelKey = vertexLabelKey)
    if not anchors:
        return []  # nothing to expand from

    anchors_lower = {a.lower() for a in anchors}

    # 2) Pull all pairs across all graphs (no graph_id filter), then filter by anchor labels in Python
    rows = manager.exec(
        """
        MATCH (a:Vertex)-[:Edge]->(b:Vertex)
        RETURN a.label AS a_label, b.label AS b_label
        """,
        {}, write=False
    ) or []

    pairs = []
    for r in rows:
        a_lab = str(r.get("a_label") or "").strip()
        b_lab = str(r.get("b_label") or "").strip()
        if a_lab and b_lab and a_lab.lower() in anchors_lower:
            pairs.append((a_lab, b_lab))

    return pairs

def candidate_counts_for_labels(manager, graph, labels: list[str], vertexLabelKey: str = "label") -> list[tuple[str,int]]:
    """Aggregate neighbor label frequencies across *all* graphs for any a.label in labels (case-insensitive)."""
    pairs = fetch_all_pairs(manager, graph = graph, vertexLabelKey=vertexLabelKey)
    label_set = {l.lower() for l in labels}
    cnt = Counter(b for (a,b) in pairs if a.lower() in label_set)
    if "" in cnt:
        del cnt[""]
    return sorted(cnt.items(), key=lambda kv: (-kv[1], kv[0]))

# ---------------------
# Seed props copier — find best example for a label across all graphs
# ---------------------

import math
from collections import Counter
from statistics import median, mean
from typing import Optional, Dict, Any

def find_best_example_for_label(manager, label: str, attach_to: str) -> Optional[Dict[str, Any]]:
    """
    Search the entire Kùzu DB for occurrences of edges (attach_to_label -> target_label).
    Use the most-popular direction (by angle bin) and typical distance (median within that bin)
    to compute a RECOMMENDED coordinate for the new node as an offset from the given attach_to node.

    Parameters
    ----------
    manager : Kùzu manager
    attach_to : str | dict
        - str: label of the attach node (e.g., "Entrance"). Anchor coords default to (0,0,0).
        - dict: must include at least {"label": "..."} and *optionally* {"x":..,"y":..,"z":..}
                If x/y/z are present, they are used as the anchor for the recommended offset.
    label_substring : str
        Target node label (first word is used for matching, e.g., "Living" from "Living Room").

    Returns
    -------
    dict or None:
      {
        "best_example": {"gid","id","label","x","y","z","props"},
        "recommended": {"x": float, "y": float, "z": float, "distance": float}
      }
      or None if no corpus matches are found.
    """
    # Normalize inputs
    if isinstance(attach_to, str):
        attach_to_label = attach_to
        anchor_x, anchor_y, anchor_z = 0.0, 0.0, 0.0
    elif isinstance(attach_to, dict):
        attach_to_label = attach_to.get("label", "")
        anchor_x = float(attach_to.get("x", 0.0))
        anchor_y = float(attach_to.get("y", 0.0))
        anchor_z = float(attach_to.get("z", 0.0))
    else:
        attach_to_label, anchor_x, anchor_y, anchor_z = "", 0.0, 0.0, 0.0

    # Only use first word (e.g., "Living" from "Living Room")
    if attach_to_label == "":
        a_word = ""
    else:
        a_word = (attach_to_label or "").split()[0].lower()
    if label == "":
        b_word = ""
    else:
        b_word = (label or "").split()[0].lower()

    # 1) Pull ALL (a -> b) pairs with coordinates
    rows = manager.exec(
        """
        MATCH (a:Vertex)-[:Edge]->(b:Vertex)
        RETURN
          a.graph_id AS agid, a.id AS aid, a.label AS a_label, a.x AS ax, a.y AS ay, a.z AS az,
          b.graph_id AS bgid, b.id AS bid, b.label AS b_label, b.x AS bx, b.y AS by, b.z AS bz, b.props AS bprops
        """,
        {}, write=False
    ) or []

    # 2) Filter in Python (case-insensitive, first-word heuristic)
    pairs = []
    for r in rows:
        a_lab = str(r.get("a_label") or "")
        b_lab = str(r.get("b_label") or "")
        if (a_word in a_lab.lower()) and (b_word in b_lab.lower()):
            ax, ay, az = float(r.get("ax", 0.0)), float(r.get("ay", 0.0)), float(r.get("az", 0.0))
            bx, by, bz = float(r.get("bx", 0.0)), float(r.get("by", 0.0)), float(r.get("bz", 0.0))
            dx, dy, dz = (bx - ax), (by - ay), (bz - az)
            dist = math.sqrt(dx*dx + dy*dy + dz*dz)
            pairs.append({
                "agid": r.get("agid"), "aid": r.get("aid"), "a_label": a_lab, "ax": ax, "ay": ay, "az": az,
                "bgid": r.get("bgid"), "bid": r.get("bid"), "b_label": b_lab, "bx": bx, "by": by, "bz": bz,
                "bprops": r.get("bprops", {}),
                "dx": dx, "dy": dy, "dz": dz, "dist": dist
            })

    if not pairs:
        return None

    # 3) Choose a "best example" node for the target label
    target_counts = Counter(p["b_label"] for p in pairs)
    best_target_label, _ = max(target_counts.items(), key=lambda kv: kv[1])
    best_row = next(p for p in pairs if p["b_label"] == best_target_label)
    best_example = {
        "gid": best_row["bgid"],
        "id":  best_row["bid"],
        "label": best_row["b_label"],
        "x": best_row["bx"], "y": best_row["by"], "z": best_row["bz"],
        "props": best_row.get("bprops", {})
    }

    # 4) Determine most-popular direction bin and typical distance
    DIR_BIN_COUNT = 16     # 22.5-degree bins
    DIST_BIN_SIZE = 0.5    # meters

    dir_bins = []
    dist_bins = []
    for p in pairs:
        ang = math.degrees(math.atan2(p["dy"], p["dx"])) % 360.0
        dir_bin = int((ang / 360.0) * DIR_BIN_COUNT) % DIR_BIN_COUNT
        dir_bins.append(dir_bin)
        dist_bins.append(int(p["dist"] / DIST_BIN_SIZE))

    # Mode direction bin and mode distance bin
    dir_mode_bin, _ = Counter(dir_bins).most_common(1)[0]

    # Compute a representative unit direction from vectors in the mode direction bin
    sel_vectors = []
    for p, db in zip(pairs, dir_bins):
        if db == dir_mode_bin and p["dist"] > 1e-9:
            ux, uy, uz = p["dx"]/p["dist"], p["dy"]/p["dist"], p["dz"]/p["dist"]
            sel_vectors.append((ux, uy, uz))

    if sel_vectors:
        mx = sum(v[0] for v in sel_vectors)/len(sel_vectors)
        my = sum(v[1] for v in sel_vectors)/len(sel_vectors)
        mz = sum(v[2] for v in sel_vectors)/len(sel_vectors)
        norm = math.sqrt(mx*mx + my*my + mz*mz) or 1.0
        unit_dir = (mx/norm, my/norm, mz/norm)
    else:
        unit_dir = (1.0, 0.0, 0.0)  # fallback

    # Typical distance: median of distances in that same direction bin (robust)
    sel_dists = [p["dist"] for p, db in zip(pairs, dir_bins) if db == dir_mode_bin]
    rec_dist = median(sel_dists) if sel_dists else 0.0

    # 5) Compute recommended coordinates as an offset from the provided attach_to anchor
    rx = anchor_x + unit_dir[0] * rec_dist
    ry = anchor_y + unit_dir[1] * rec_dist
    rz = anchor_z + unit_dir[2] * rec_dist

    return {
        "best_example": best_example,
        "recommended": {"x": rx, "y": ry, "z": rz, "distance": rec_dist}
    }

# ---------------------
# Global-candidate logic + LLM action picker — single action per iteration
# ---------------------

import requests

@dataclass
class LLMConfig:
    provider: str                   # "openai" | "gemini" | "ollama" | "lmstudio"
    model: str                      # e.g. "gpt-4o-mini", "gemini-1.5-pro", "llama3.1", etc.
    api_key: Optional[str] = None   # OPENAI_API_KEY / GOOGLE_API_KEY as applicable
    base_url: Optional[str] = None  # for LM Studio (OpenAI-compatible) or custom Ollama host
    temperature: float = 0.2
    timeout: int = 60               # seconds
    max_output_tokens: Optional[int] = None

# OpenAI (requires OPENAI_API_KEY env var or pass api_key)
cfg_openai = LLMConfig(
    provider="openai",
    model="gpt-5",
    api_key=os.getenv("OPENAI_API_KEY"),
)

# Claude (requires ANTHROPIC_API_key env var or pass api_key)
cfg_claude = LLMConfig(
    provider="claude",
    model="claude-haiku-4-5-20251001",
    api_key=os.getenv("ANTHROPIC_API_KEY"),
)
# Gemini (requires GOOGLE_API_KEY env var or pass api_key)
cfg_gemini = LLMConfig(
    provider="gemini",
    model="gemini-1.5-pro",          # or "gemini-1.5-flash"
    api_key=os.getenv("GOOGLE_API_KEY"),
)

# Ollama (local)
cfg_ollama = LLMConfig(
    provider="ollama",
    model="llama3.1",                # or "mistral", "qwen2.5", etc.
    base_url="http://localhost:11434",
)

# LM Studio (local OpenAI-compatible; default server on :1234)
cfg_lmstudio = LLMConfig(
    provider="lmstudio",
    model="TheBloke/Mistral-7B-Instruct-GGUF",  # whatever model you've loaded in LM Studio
    base_url="http://localhost:1234/v1",
)

def coerce_json(text: str) -> Dict[str, Any]:
    import re
    if not text: return {}
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip(), flags=re.DOTALL)
    m = re.search(r"\{.*\}", text, re.DOTALL)
    blob = m.group(0) if m else text
    try: return json.loads(blob)
    except Exception: return {}

def call_llm_json(prompt_system: str, payload_user: Dict[str, Any], cfg: LLMConfig) -> Dict[str, Any]:
    """
    Call a chat model and return parsed JSON dict.
    Supported providers: "openai", "gemini", "ollama", "lmstudio".
    """
    # Common messages structure (used by OpenAI, LM Studio, Ollama)
    messages = [
        {"role": "system", "content": prompt_system},
        {"role": "user", "content": json.dumps(payload_user)},
    ]

    if cfg.provider == "openai":
        try:
            from openai import OpenAI
            client = OpenAI()
            resp = client.chat.completions.create(
                model="gpt-5",
                messages=[{"role":"system","content":prompt_system},
                            {"role":"user","content":json.dumps(payload_user)}],
            )
            text = resp.choices[0].message.content.strip()
            c_j = coerce_json(text)
            return c_j
        except Exception:
            return {}

    # --- Claude / Anthropic ---
    if cfg.provider == "anthropic":
        # try:
        from anthropic import Anthropic
        client = Anthropic(api_key=cfg.api_key or os.getenv("ANTHROPIC_API_KEY"))
        resp = client.messages.create(
            model=cfg.model,
            system=prompt_system,
            messages=[{"role":"user","content":json.dumps(payload_user)}],
            temperature=cfg.temperature,
            max_tokens=cfg.max_output_tokens or 512,
        )
        # Claude 3 returns message.content as list of parts
        parts = resp.content
        text = "".join(p.text for p in parts if hasattr(p, "text"))
        return coerce_json(text)
        # except Exception:
        #    return {}
    
    prompt_system


    if cfg.provider == "google":
        try:
            import google.generativeai as genai

            # Configure Gemini API key (from cfg or environment)
            genai.configure(api_key=cfg.api_key or os.getenv("GEMINI_API_KEY"))

            # Combine the system and user prompts into a single text prompt
            prompt = (
                f"{prompt_system.strip()}\n\n"
                "User payload (as JSON):\n"
                f"{json.dumps(payload_user, indent=2)}\n\n"
                "Return STRICT JSON only."
            )

            # Create model instance
            model = genai.GenerativeModel(cfg.model or "gemini-1.5-pro")

            # Generate content
            resp = model.generate_content(
                prompt,
                generation_config={
                    "temperature": cfg.temperature if hasattr(cfg, "temperature") else 0.2,
                    "max_output_tokens": getattr(cfg, "max_output_tokens", 512),
                }
            )

            # Extract the response text (Gemini returns a 'GenerativeResponse' object)
            text = getattr(resp, "text", "") or (
                resp.candidates[0].content.parts[0].text
                if getattr(resp, "candidates", None) else ""
            )

            text = text.strip()
            c_j = coerce_json(text)
            return c_j

        except Exception as e:
            print("Gemini error:", e)
            return {}

    if cfg.provider == "ollama":
        print("Provider is Ollama (local)")
        try:
            url = (cfg.base_url or "http://localhost:11434").rstrip("/") + "/api/chat"
            r = requests.post(url, json={
                "model": cfg.model,
                "messages": messages,
                "stream": False,
                "options": {
                    "temperature": cfg.temperature,
                    **({} if cfg.max_output_tokens is None else {"num_predict": cfg.max_output_tokens})
                }
            }, timeout=cfg.timeout)
            r.raise_for_status()
            data = r.json()
            # Newer Ollama returns {"message":{"content": ...}}
            if "message" in data:
                text = data["message"].get("content", "")
            else:
                text = data.get("choices", [{}])[0].get("message", {}).get("content", "")
            return coerce_json(text)
        except Exception:
            return {}

    if cfg.provider == "lmstudio":
        try:
            # LM Studio provides an OpenAI-compatible server (default http://localhost:1234/v1)
            base = (cfg.base_url or "http://localhost:1234/v1").rstrip("/")
            url = base + "/chat/completions"
            r = requests.post(url, json={
                "model": cfg.model,
                "messages": messages,
                "temperature": cfg.temperature,
                **({} if cfg.max_output_tokens is None else {"max_tokens": cfg.max_output_tokens})
            }, timeout=cfg.timeout)
            r.raise_for_status()
            text = r.json()["choices"][0]["message"]["content"]
            return coerce_json(text)
        except Exception:
            return {}

    return {}

# -----------------------------------------------------
# HEURESTIC ACTION IF NO LLM

from collections import defaultdict
from collections import Counter
from typing import Dict, Iterable, Tuple

def corpus_label_frequencies(
    manager,
    *,
    include_graph_ids: Iterable[str] | None = None,
    exclude_graph_ids_prefixes: Tuple[str, ...] = ("work_",),
    first_word_only: bool = True,
    normalize_case: bool = True,
) -> Dict[str, int]:
    """
    Compute corpus label frequencies from all Vertex nodes in Kùzu.

    Returns
    -------
    dict[str, int]
        Mapping from label (possibly normalised) to frequency across the corpus.
    """
    # 1) Pull all graph_ids + labels once
    rows = manager.exec(
        """
        MATCH (v:Vertex)
        RETURN v.graph_id AS gid, v.label AS label
        """,
        {},
        write=False,
    ) or []

    def graph_allowed(gid: str) -> bool:
        if include_graph_ids is not None:
            return gid in include_graph_ids
        return not any(gid.startswith(pref) for pref in exclude_graph_ids_prefixes)

    counter = Counter()
    for r in rows:
        gid = r.get("gid", "")
        if not graph_allowed(gid):
            continue

        label = str(r.get("label") or "")

        if first_word_only:
            label = label.split()[0] if label else ""

        if normalize_case:
            label = label.strip().lower()

        if label:
            counter[label] += 1

    return dict(counter)

def corpus_pair_frequencies(
    manager,
    *,
    include_graph_ids: Iterable[str] | None = None,
    exclude_graph_ids_prefixes: Tuple[str, ...] = ("work_",),
    first_word_only: bool = True,
    normalize_case: bool = True,
) -> Dict[Tuple[str, str], int]:
    """
    Compute corpus frequencies of unordered label–label adjacency pairs
    from all edges in the Kùzu database.

    Returns
    -------
    dict[(label_a, label_b), int]
        Mapping from unordered label-pair to frequency across the corpus.
    """

    # 1. Fetch all edges with labels of endpoints
    rows = manager.exec(
        """
        MATCH (a:Vertex)-[:Edge]->(b:Vertex)
        RETURN 
            a.graph_id AS agid, a.label AS a_label,
            b.graph_id AS bgid, b.label AS b_label
        """,
        {},
        write=False,
    ) or []

    def graph_allowed(gid: str) -> bool:
        if include_graph_ids is not None:
            return gid in include_graph_ids
        return not any(gid.startswith(pref) for pref in exclude_graph_ids_prefixes)

    pair_counter = Counter()

    for r in rows:
        gid = r.get("agid")
        if not graph_allowed(gid):
            continue

        a_label = str(r.get("a_label") or "")
        b_label = str(r.get("b_label") or "")
        if not a_label or not b_label:
            continue

        # Normalise labels
        if first_word_only:
            a_label = a_label.split()[0]
            b_label = b_label.split()[0]

        if normalize_case:
            a_label = a_label.lower().strip()
            b_label = b_label.lower().strip()

        # Ignore invalid pairs
        if not a_label or not b_label:
            continue

        # Unordered pair
        if a_label <= b_label:
            pair = (a_label, b_label)
        else:
            pair = (b_label, a_label)

        pair_counter[pair] += 1

    return dict(pair_counter)

def _normalize_pair(a: str, b: str) -> tuple[str, str]:
    return (a, b) if a <= b else (b, a)

def _edge_set_from_current(current_edges, id_to_label):
    """
    Returns a set of unordered label-pairs that are already connected in the current graph,
    regardless of direction and which specific node instances are used.
    """
    edge_set = set()
    if not current_edges:
        return edge_set

    # Support both [(a,b)] and [{'a':..., 'b':...}, ...]
    for e in current_edges:
        if isinstance(e, dict):
            a_id, b_id = e.get("a"), e.get("b")
        else:
            a_id, b_id = e  # tuple
        la = id_to_label.get(a_id)
        lb = id_to_label.get(b_id)
        if la and lb and la != "" and lb != "":
            edge_set.add(_normalize_pair(la, lb))
    return edge_set

def _degree_by_node(current_edges):
    """
    Simple undirected degree per node-id for tie-breaking deletes / attachments.
    Supports both list[tuple] and list[dict].
    """
    deg = defaultdict(int)
    for e in current_edges or []:
        if isinstance(e, dict):
            a, b = e.get("a"), e.get("b")
        else:
            a, b = e
        if a is None or b is None: 
            continue
        deg[a] += 1
        deg[b] += 1
    return deg

def heuristic_pick_action(
    *,
    current_nodes: list[dict],
    current_edges: list,
    candidate_counts: list[tuple[str,int]],
    corpus_label_freq: dict[str, int],
    corpus_pair_freq: dict[tuple[str, str], int],
    min_label_freq_for_keep: int = 1,     # labels below this are considered for deletion
    min_pair_freq_for_connect: int = 2,   # only consider connects at/above this frequency
    top_k_labels_to_consider: int = 10,   # look at top-K missing labels for ADD
    top_k_pairs_to_consider: int = 20     # look at top-K pairs for CONNECT
) -> dict:
    """
    Returns ONE of:
      {"action":"add","new_label":"<label>","attach_to":"<existing_node_id>"}
      {"action":"connect","a":"<node_id>","b":"<node_id>"}
      {"action":"delete","node_id":"<node_id>","label":"<label>","reason":"<string>"}
      {"action":"stop","reason":"<string>"}
    """
    print("Using heuristics. Not LLMs.")
    # --- Build quick lookups
    id_to_label = {n["id"]: n["label"] for n in current_nodes}
    label_to_ids = defaultdict(list)
    for n in current_nodes:
        label_to_ids[n["label"]].append(n["id"])

    labels_present = set(label_to_ids.keys())
    edge_labels_present = _edge_set_from_current(current_edges, id_to_label)
    degree = _degree_by_node(current_edges)

    # ---- 1) ADD: highest-frequency label not yet present
    # prefer candidate_counts (already frequency-sorted from corpus), but double-check against corpus_label_freq
    if candidate_counts:
        for lab, _cnt in candidate_counts[:top_k_labels_to_consider]:
            if lab not in labels_present:
                # choose attach_to: the node whose label has strongest pair-frequency with `lab`
                best_attach_id = None
                best_pair_score = -1
                for existing_lab in labels_present:
                    pair = _normalize_pair(lab, existing_lab)
                    score = corpus_pair_freq.get(pair, 0)
                    if score > best_pair_score and label_to_ids[existing_lab]:
                        # pick the lowest-degree instance of that label to reduce hubs
                        candidates = label_to_ids[existing_lab]
                        best_attach_id = min(candidates, key=lambda nid: degree.get(nid, 0))
                        best_pair_score = score

                # fallback: if no pairs known, attach to lowest-degree node overall (if any)
                if not best_attach_id and current_nodes:
                    best_attach_id = min((n["id"] for n in current_nodes), key=lambda nid: degree.get(nid, 0))

                if best_attach_id:
                    return {"action": "add", "a": lab, "b": best_attach_id}
                else:
                    # no nodes yet: seed case
                    return {"action": "seed", "a": lab}

    # ---- 2) CONNECT: pick a high-frequency label–label pair that is missing
    # Build a list of candidate pairs among labels present, scored by corpus frequency
    if labels_present and len(labels_present) >= 2:
        # sort pairs by corpus freq desc
        # generate all unordered pairs of labels present
        present_list = sorted(labels_present)
        scored_pairs = []
        for i in range(len(present_list)):
            for j in range(i+1, len(present_list)):
                pair = _normalize_pair(present_list[i], present_list[j])
                f = corpus_pair_freq.get(pair, 0)
                if f >= min_pair_freq_for_connect:
                    scored_pairs.append((pair, f))
        scored_pairs.sort(key=lambda x: x[1], reverse=True)

        for (la, lb), f in scored_pairs[:top_k_pairs_to_consider]:
            if _normalize_pair(la, lb) not in edge_labels_present:
                # choose specific node instances to connect
                if label_to_ids[la] and label_to_ids[lb]:
                    # connect lowest-degree instances to avoid hubs
                    a_id = min(label_to_ids[la], key=lambda nid: degree.get(nid, 0))
                    b_id = min(label_to_ids[lb], key=lambda nid: degree.get(nid, 0))
                    if a_id != b_id:
                        return {"action": "connect", "a": a_id, "b": b_id}

    # ---- 3) DELETE: any node whose label is very low-freq or absent in the corpus
    # Find labels with corpus freq < threshold (or not present at all)
    low_labels = [lab for lab in labels_present if corpus_label_freq.get(lab, 0) < min_label_freq_for_keep]
    if low_labels:
        # choose the lowest-degree node among all low-frequency labels
        low_nodes = []
        for lab in low_labels:
            for nid in label_to_ids[lab]:
                low_nodes.append((nid, lab, degree.get(nid, 0)))
        if low_nodes:
            nid, lab, _deg = min(low_nodes, key=lambda t: t[2])
            return {
                "action": "delete",
                "node_id": nid,
                "label": lab,
                "reason": f"Label '{lab}' has corpus frequency {corpus_label_freq.get(lab,0)} < {min_label_freq_for_keep}"
            }

    # ---- 4) Otherwise: STOP
    return {"action": "stop", "reason": "No high-frequency additions or connections missing; no low-frequency labels to prune."}


# -----------------------------------------------------



def llm_pick_action(description,
                    current_nodes: list[Dict[str,str]],
                    candidate_counts: list[tuple[str,int]],
                    current_edges: list,
                    actions_log: list,
                    cfg: LLMConfig
                    ):
    """
    Ask the LLM to choose exactly one action. It knows the candidate list is frequency-sorted
    but may propose a new label not in the list. Returns one of:
      {"action":"seed","a":"Kitchen"}
      {"action":"add","a":"Kitchen","b":"<existing_local_id> ("Dining)"}
      {"action":"delete","a":"Kitchen"}
      {"action":"connect","a":"<existing_local_id>","b":"<existing_local_id>"}
      {"action":"disconnect","a":"<existing_local_id>","b":"<existing_local_id>"}
    """
    import copy
    # if (openai is None) or (os.getenv("OPENAI_API_KEY") is None):
    #     return _heuristic_pick_action(current_nodes, candidate_counts)

    # openai.api_key = os.environ["OPENAI_API_KEY"]
    sys_prompt = (
        f"You are designing an adjacency graph that represents {description}. You receive: "
        "(1) A description of what the graph represents, (2) the current graph's nodes, (2) the current graph's edges, (3) a list of previous actions, and (4) a frequency-sorted list of candidate neighbor labels "
        f"aggregated from many example graphs. Build a list of node labels usually found in a graph that represents {description}."
        "You may choose from the provided list of candidate node labels or propose a new label from the list that you built."
        "Choose exactly ONE action: either SEED a new node with no connections, ADD a new node with a single connection to an existing node, "
        "DELETE an existing node, or CONNECT two existing nodes, "
        "or DISCONNECT two existing nodes, or STOP if no further action is needed. Include a reason for stopping."
        "Do not repeat actions that have been flagged as 'accepted=False'."
        "Return strict JSON with one of the forms:\n"
        "{\"action\":\"seed\",\"a\":\"<string>\"}\n"
        "{\"action\":\"add\",\"a\":\"<string>\",\"b\":\"<existing_local_id> (<string>)\"}\n"
        "{\"action\":\"delete\",\"a\":\"<existing_local_id> (<string>)\"}\n"
        "{\"action\":\"connect\",\"a\":\"<existing_local_id> (<string>) \",\"b\":\"<existing_local_id> (<string>)\"}"
        "{\"action\":\"disconnect\",\"a\":\"<existing_local_id> (<string>) \",\"b\":\"<existing_local_id> (<string>)\"}"
        "{\"action\":\"stop\",\"reason\":\"<string> \"}"
    )
    user_payload = {
        "description": description, # A description of the type of thing to be created
        "current_nodes": current_nodes,                 # list of {id,label,props}
        "current_edges": current_edges or [],   # list of edge connection
        "candidate_counts": candidate_counts,           # list of [label, count], sorted desc
        "actions_log": actions_log or [],
        "note": "The candidate list is sorted by frequency across many graphs; you may propose a new label."
    }

    json_data = call_llm_json(prompt_system = sys_prompt, payload_user = user_payload, cfg=cfg)
    return json_data

# ---------------------
# Builder loop — seed from dataset example, then iterate
# ---------------------

def import_folder_to_kuzu(json_folder: str, manager, labelKey: str, undirected: bool = True) -> List[str]:
    graph_ids: List[str] = []
    for path in sorted(glob.glob(os.path.join(json_folder, "*.json"))):
        verts, edges = load_topologic_graph(path, labelKey)
        gid = os.path.splitext(os.path.basename(path))[0]
        upsert_graph(manager, gid, verts, edges, undirected=undirected)
        graph_ids.append(gid)
    return graph_ids


def ask_user_action(prompt: str) -> str:
    """
    Ask the user to choose one of three actions: add / connect / stop.
    Keeps prompting until a valid choice is entered.

    Returns
    -------
    str : one of "accept", "ignore", or "stop"
    """
    valid = {"accept", "ignore", "stop"}
    while True:
        choice = input(prompt+" Choose an action [accept / ignore / stop]: ").strip().lower()
        if choice == "":
            choice = "accept"
            return choice
        if choice == "stop":
            print("→ You chose to stop the process.")
            return choice
        if choice in valid:
            print(f"→ Suggestion: {prompt}")
            print(f"→ You chose to {choice} this suggestion")
            return choice
        print("Invalid choice. Please type one of: accept, ignore, or stop.")

@dataclass
class LLMConfig:
    provider: str                   # "openai" | "gemini" | "ollama" | "lmstudio"
    model: str                      # e.g. "gpt-4o-mini", "gemini-1.5-pro", "llama3.1", etc.
    api_key: Optional[str] = None   # OPENAI_API_KEY / GOOGLE_API_KEY as applicable
    base_url: Optional[str] = None  # for LM Studio (OpenAI-compatible) or custom Ollama host
    temperature: float = 0.2
    timeout: int = 60               # seconds
    max_output_tokens: Optional[int] = None

@staticmethod
def Generate(graph,
           vertexLabelKey: str = "label",
           description: str = "",
           manager = None,
           maxSteps: int = 8,
           patience: int = 2,
           provider: str = "openai", # "openai" | "google" | "anthropic" | "ollama" | "lmstudio"
           model: str = "gpt-5", # e.g. "gpt-5", "gpt-4o-mini", "gemini-1.5-pro", "llama3.1", etc.
           apiKey: Optional[str] = None,   # OPENAI_API_KEY / GOOGLE_API_KEY as applicable
           baseURL: Optional[str] = None,  # for LM Studio (OpenAI-compatible) or custom Ollama host
           temperature: float = 0.2,
           timeout: int = 60,           # seconds
           maxOutputTokens: Optional[int] = None,
           automatic: bool = False
           ) -> Dict[str,Any]:
    """
    DocString to be added
    """
    from topologicpy.Dictionary import Dictionary
    from topologicpy.Topology import Topology
    import copy
    import random


    llm_config = LLMConfig(provider=provider,
                           model=model,
                           api_key=apiKey,
                           base_url=baseURL,
                           temperature=temperature,
                           timeout=timeout,
                           max_output_tokens=maxOutputTokens)
    
    # Assign a unique sequential integer id to each vertex
    vertices = Graph.Vertices(graph)
    for i, v in enumerate(vertices):
        d = Topology.Dictionary(v)
        d = Dictionary.SetValueAtKey(d, "id", str(i))
        vertex_label = Dictionary.ValueAtKey(d, vertexLabelKey, "Unknown")
        best_ex_dict = find_best_example_for_label(manager, label = vertex_label, attach_to = None)
        if best_ex_dict is not None:
            best_ex = best_ex_dict.get("best_example", None)
            if best_ex is not None:
                vertex_label = best_ex.get("label", vertex_label)
        d = Dictionary.SetValueAtKey(d, vertexLabelKey, vertex_label)
        v = Topology.SetDictionary(v, d)

    snapshots = [graph]
    actions_log: list[Dict[str,Any]] = []

    no_action = 0
    for step in range(1, maxSteps+1):
        print("STEP:", step)
        
        current_nodes, current_edges = list_working_nodes_edges(graph, vertexLabelKey)
        labels_now = [n[vertexLabelKey] for n in current_nodes]
        cand_counts = candidate_counts_for_labels(manager=manager,
                                                  graph=graph,
                                                  labels=labels_now,
                                                  vertexLabelKey=vertexLabelKey)

        if provider == "heuristic":
            print("User selected heuristic")
            corpus_label_freq = corpus_label_frequencies(manager)
            corpus_pair_freq = corpus_pair_frequencies(manager)
            action = heuristic_pick_action( current_nodes=current_nodes,             # [{'id': 'n0', 'label': 'Entrance', 'props': {...}}, ...]
                                           current_edges=current_edges,             # either [{'a':'n0','b':'n1',...}, ...] or [('n0','n1'), ...]
                                           candidate_counts=cand_counts,       # [('Living', 123), ('Kitchen', 117), ...] sorted desc
                                           corpus_label_freq=corpus_label_freq,           # {'Living': 200, 'Kitchen': 195, ...}
                                           corpus_pair_freq=corpus_pair_freq,             # {('Entrance','Living'): 120, ('Living','Kitchen'): 110, ...}
                                           min_label_freq_for_keep=1,               # tune as you like
                                           min_pair_freq_for_connect=2             # tune as you like
            )
            json_action = action.get('action', 'Unknown')
            print("json_action:", json_action)
            json_a_label = action.get('a')
            json_b_label = action.get('b')
            print("json_a_label:", json_a_label)
            print("json_b_label:", json_b_label)
            if json_b_label is not None:
                print_b_label = copy.copy(json_b_label)
            else:
                print_b_label = "Unknown"
        else:
            action = llm_pick_action(description = description,
                                    current_nodes = current_nodes,
                                    candidate_counts = cand_counts,
                                    current_edges= current_edges,
                                    actions_log = actions_log,
                                    cfg=llm_config)
        
            json_action = action.get('action', 'Unknown')
            print("json_action:", json_action)
            json_a_label = action.get('a')
            json_b_label = action.get('b')
            print("json_a_label:", json_a_label)
            print("json_b_label:", json_b_label)
            if json_b_label is not None:
                print_b_label = copy.copy(json_b_label)
                print_b_label = print_b_label.split()[1].strip("()")
            else:
                print_b_label = "Unknown"
        if "seed" in json_action.lower():
            prompt = f" I suggest that you {json_action.lower()} '{json_a_label}'"
        elif "add" in json_action.lower():
            prompt = f" I suggest that you {json_action.lower()} '{json_a_label}' and connect it to '{print_b_label}'"
        elif "delete" in json_action.lower():
            prompt = f" I suggest that you {json_action.lower()} '{json_a_label}'"
        elif "connect" in json_action.lower():
            prompt = f" I suggest that you {json_action.lower()} '{json_a_label}' to '{print_b_label}'"
        elif "disconnect" in json_action.lower():
            prompt = f" I suggest that you {json_action.lower()} '{json_a_label}' from '{print_b_label}'"
        elif "stop" in json_action.lower():
            prompt = " I suggest that you stop."
        else:
            prompt = " I don't know what else to suggest."
        # Check for Human-in-the-loop
        if automatic == False:
            answer = ask_user_action(prompt)
            if answer == "stop":
                break
            if answer == "ignore":
                action['accepted'] = False
                actions_log.append(action)
                continue
            else:
                action['accepted'] = True
        else:
            print(prompt)
        if action.get("action") == "seed":
            a_label = str(action.get("a") or "").strip()
            if a_label:
                #new_id = f"n{len(current_nodes)}"
                # attempt to copy props from best example for a_label
                best_ex_dict = find_best_example_for_label(manager, label = a_label, attach_to=None)
                if best_ex_dict is not None:
                    ex = best_ex_dict['best_example']
                else:
                    ex = None
                
                props = {}
                if ex is not None:
                    a_label = ex.get(vertexLabelKey, ex.get("label",a_label))
                    props = ex.get("props", {})
                    if isinstance(props, str):
                        try: props = json.loads(props)
                        except Exception: props = {"_raw_props": props}
                    props = dict(props or {})
                    props.update({
                        "source": "suggested_node_from_dataset",
                        "matched_label": ex.get("label",""),
                        "matched_graph_id": ex.get("gid",""),
                        "matched_vertex_id": ex.get("id",""),
                    })
                    x = best_ex_dict['recommended']['x']
                    y = best_ex_dict['recommended']['y']
                    z = best_ex_dict['recommended']['z']
                else:
                    x = random.Uniform(0,100)
                    y = random.Uniform(0,100)
                    z = random.Uniform(0,100)
                    props = {"source": "suggested_node_no_example"}
                a_id = unique_id(graph)
                create_vertex(graph, id= a_id, label=a_label, vertexLabelKey=vertexLabelKey, props=props, x=x, y=y, z=z)
            else:
                no_action += 1
        elif action.get("action") == "add":
            a_label = str(action.get("a") or "").strip()
            b_id = str(action.get("b") or "").strip().split()[0]
            # ensure b_label is a valid existing local id
            existing_ids = [str(n["id"]) for n in current_nodes]
            if b_id not in existing_ids:
                print(f"Warning: Could not find node id {b_id} in the working graph")
                b_id = None
            if a_label and b_id:
                # attempt to copy props from best example for new_label
                best_ex_dict = find_best_example_for_label(manager, label = a_label, attach_to = None)
                if best_ex_dict is not None:
                    ex = best_ex_dict['best_example']
                else:
                    ex = None
                
                props = {}
                # x = y = z = 0.0
                if ex is not None:
                    a_label = ex.get(vertexLabelKey, ex.get("label",a_label))
                    props = ex.get("props", {})
                    if isinstance(props, str):
                        try: props = json.loads(props)
                        except Exception: props = {"_raw_props": props}
                    props = dict(props or {})
                    props.update({
                        "source": "suggested_node_from_dataset",
                        "matched_label": ex.get("label",""),
                        "matched_graph_id": ex.get("gid",""),
                        "matched_vertex_id": ex.get("id",""),
                    })
                    x = best_ex_dict['recommended']['x']
                    y = best_ex_dict['recommended']['y']
                    z = best_ex_dict['recommended']['z']
                else:
                    x_min, y_min, x_max, y_max = aabb(graph)
                    x = random.uniform(x_min, x_max)
                    y = random.uniform(y_min, y_max)
                    z = 0.0
                    props = {"source": "suggested_node_no_example"}
                a_id = unique_id(graph)
                new_graph = create_vertex(graph, id = a_id, label=a_label, vertexLabelKey=vertexLabelKey, props=props, x=x, y=y, z=z)
                if new_graph is not None:
                    graph = new_graph
                    new_graph = create_edge(graph, a=a_id, b=b_id, label="suggested")
                if new_graph is not None:
                    graph = new_graph
            else:
                no_action += 1
        
        elif action.get("action") == "disconnect":
            a_id = str(action.get("a") or "").strip()
            b_id = str(action.get("b") or "").strip().split()[0]
            # ensure b_label is a valid existing local id;
            existing_ids = {n["id"] for n in current_nodes}
            if b_id not in existing_ids:
                print(f"Warning 2: Could not find node id {b_id} in the working graph")
            if action.get("action") == "disconnect":
                if a_id not in existing_ids:
                    print(f"Warning 3: Could not find node id {a_id} in the working graph")
                    #a_id = next(iter(existing_ids), None)
            if a_id and b_id:
                applied = delete_edge(graph, a_id, b_id)
                if not applied:
                    no_action +=1
            else:
                no_action += 1

        elif action.get("action") == "connect":
            a = str(action.get("a") or "").strip()
            b = str(action.get("b") or "").strip()
            if a and b and a != b:
                applied = create_edge(graph, a, b, label="suggested")
                if not applied:
                    no_action +=1
        elif action.get("action") == "delete":
            a_id = str(action.get("a") or "").strip().split()[0]
            if a_id:
                applied = delete_vertex(graph, a_id)
                if not applied:
                    no_action +=1

        elif action.get("action") == "stop":
            print("Instructed to stop. Stopping.")
            actions_log.append(action)
            snapshots.append(graph)
            return {"snapshots": snapshots, "actions": actions_log, "reason": "Action produced no change. Ran out of patience"}
        
        if no_action > patience:
            print("Ran out of patience with no action. Stopping.")
            return {"snapshots": snapshots, "actions": actions_log, "reason": "Action produced no change. Ran out of patience"}

        
        actions_log.append(action)
        snapshots.append(graph)

    return {"snapshots": snapshots, "actions": actions_log, "reason": f"Reached max steps ({maxSteps})."}


## Create a Kuzu DB Manager

In [ ]:
db_path = "C:/Users/sarwj/OneDrive - Cardiff University/Desktop/demo_kuzu"         # Kùzu DB directory (will be created/used)
mgr = Kuzu.Manager(db_path)

## Import the graphs and store in Kuzu (Run Once)
### Replace path name with your path
### Specify labelKey = "node_name" for MSD and "type" for ResPlan

In [ ]:
#json_folder = r"C:\Users\sarwj\OneDrive - Cardiff University\Desktop\msd_json/msd_sample_graphs"        # folder with your *.json graphs
json_folder = r"C:\Users\sarwj\OneDrive - Cardiff University\Desktop\resplan_json\resplan_sample_graphs"
_ = Kuzu.EmptyDatabase(mgr,  recreateSchema = False)
gids = import_folder_to_kuzu(json_folder, mgr, labelKey="type", undirected=True)
print("Imported", len(gids), "graphs")


## Create an Initial Graph

In [ ]:
import random
working_graph_id = "work_demo"
labels = ["Entry", "Conference Room", "Living Room"]
coords = [[0,0,0], [0,1,0], [1,0,0]]

dictionaries = []
for i, label in enumerate(labels):
    d = Dictionary.ByKeysValues(["id","label"], [str(i),label])
    dictionaries.append(d)

edges = [[0,1], [0,2], [1,2]]


initial_graph = Graph.ByMeshData(coords, edges, vertexDictionaries=dictionaries)

Topology.Show(initial_graph, vertexSize=12, showVertexLabel=True, vertexLabelKey="label", backgroundColor="white", width=400, height=400)




## Generate (expand) the Graph
# Try Automatic and Manual modes (Human-in-the-loop.. Look up for dialog box in manual)

In [ ]:
import os
providers = ["heuristic", "openai", "google", "anthropic", "ollama"]
models = [None, "gpt-5", "gemini-2.5-pro", "claude-haiku-4-5", "llama3.1"]
api_keys = [None, os.environ['OPENAI_API_KEY'], os.environ['GOOGLE_API_KEY'], os.environ['ANTHROPIC_API_KEY'], None]
n = 0
# Build a new working graph from a seed label that copies props from dataset
result = Generate(graph=initial_graph,
                description = "4 bedroom apartment with home office and a nursery",
                vertexLabelKey="label",
                manager=mgr,
                maxSteps = 10,
                patience = 4,
                provider = providers[n],
                model = models[n],
                apiKey = api_keys[n],
                baseURL = "http://localhost:11434",  # for LM Studio (OpenAI-compatible) or custom Ollama host
                temperature = 0.2,
                timeout = 60, # seconds
                maxOutputTokens = None,
                automatic=True)
print(result["reason"])    # why it stopped
result["actions"]           # actions chosen at each step

## The Final Resulting Graph

In [ ]:

last_graph = result["snapshots"][-1]  # TopologicPy Graph representation of the last graph
verts = Graph.Vertices(last_graph)
for v in verts:
    d = Topology.Dictionary(v)
    keys = Dictionary.Keys(d)
    if "node_name" in keys:
        d = Dictionary.SetValueAtKey(d, "label", Dictionary.ValueAtKey(d, "node_name"))
        v = Topology.SetDictionary(v, d)
Topology.Show(Graph.Reshape(last_graph),
              backgroundColor="white",
              vertexLabelKey="label",
              showVertexLabel=True,
              vertexSize=10,
              width=400,
              height=400,
              camera=[0,0,4])